In [ ]:
"""Figure 6E-F: Synapse proximity to tracheole terminal ends.

Inputs: CATMAID nodes for 117 tracheoles, FAFB v783 Princeton
presynaptic contacts, and FlyWire neuron neurotransmitter annotations.
Distances are in µm. NT confidence >= 0.5; 0.05-µm ECDF bins and 1-µm
chi-square bins with distances >= 50 µm pooled in one tail bin.
"""

from pathlib import Path
import math
import re
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import chi2_contingency

folder = Path.home() / "Downloads"
node_csv = folder / "catmaid_nodes.csv"
syn_csv = folder / "fafb_v783_princeton_synapse_table.csv"
nt_csv = folder / "neurons.csv"
out_dir = folder / "fig6"
out_dir.mkdir(parents=True, exist_ok=True)

spacing_nm, nt_min_score, chunk_size, workers = 100.0, 0.5, 250_000, -1
hist_step, hist_max = 0.05, 100.0
fine_edges = np.arange(0.0, hist_max + hist_step * 0.5, hist_step, dtype=np.float64)
hist_edges = np.append(fine_edges, np.inf)
n_hist_bins = len(hist_edges) - 1

syn_root_cols = ['pre_root_id_720575940', 'pre_root_id', 'presynaptic_root_id', 'pre_pt_root_id']
syn_x_cols = ['pre_x', 'presynaptic_x', 'pre_pt_position_x']
syn_y_cols = ['pre_y', 'presynaptic_y', 'pre_pt_position_y']
syn_z_cols = ['pre_z', 'presynaptic_z', 'pre_pt_position_z']
nt_root_cols = ['root_id', 'root', 'flywire_root_id', 'segment_id']
nt_type_cols = ['nt_type', 'transmitter', 'neurotransmitter']
nt_score_cols = ['nt_type_score', 'transmitter_prob', 'transmitter_probability', 'nt_score']
skid_cols = ['skeleton_id', 'new_skeleton_id', 'skid', 'neuron']
node_cols = ['node_id', 'treenode_id', 'id']
parent_cols = ['parent_id', 'parent_node_id', 'parent']
trach_x_cols = ['x_nm', 'x']
trach_y_cols = ['y_nm', 'y']
trach_z_cols = ['z_nm', 'z']

def normalise_name(value):
    return re.sub('[^a-z0-9]+', '_', str(value).strip().lower()).strip('_')

def resolve_column(columns, candidates):
    lookup = {normalise_name(column): column for column in columns}
    for candidate in candidates:
        key = normalise_name(candidate)
        if key in lookup:
            return lookup[key]

def clean_text(series):
    return series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

def clean_root_id(series):
    return series.astype('string').str.strip().str.replace('\\.0$', '', regex=True).replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

def load_trachea_nodes(path):
    raw = pd.read_csv(path, low_memory=False)
    columns = list(raw.columns)
    skid_col = resolve_column(columns, skid_cols)
    node_col = resolve_column(columns, node_cols)
    parent_col = resolve_column(columns, parent_cols)
    x_col = resolve_column(columns, trach_x_cols)
    y_col = resolve_column(columns, trach_y_cols)
    z_col = resolve_column(columns, trach_z_cols)
    nodes = pd.DataFrame({'skeleton_id': pd.to_numeric(raw[skid_col], errors='coerce'), 'node_id': pd.to_numeric(raw[node_col], errors='coerce'), 'parent_id': pd.to_numeric(raw[parent_col], errors='coerce'), 'x_nm': pd.to_numeric(raw[x_col], errors='coerce'), 'y_nm': pd.to_numeric(raw[y_col], errors='coerce'), 'z_nm': pd.to_numeric(raw[z_col], errors='coerce')})
    nodes = nodes.dropna(subset=['skeleton_id', 'node_id', 'x_nm', 'y_nm', 'z_nm']).copy()
    nodes['skeleton_id'] = nodes['skeleton_id'].astype(np.int64)
    nodes['node_id'] = nodes['node_id'].astype(np.int64)
    nodes['parent_id'] = pd.to_numeric(nodes['parent_id'], errors='coerce').astype('Int64')
    nodes = nodes.drop_duplicates(['skeleton_id', 'node_id']).reset_index(drop=True)
    n_skeletons = int(nodes['skeleton_id'].nunique())
    return nodes

def load_nt_lookup(path):
    header = pd.read_csv(path, nrows=0)
    columns = list(header.columns)
    root_col = resolve_column(columns, nt_root_cols)
    type_col = resolve_column(columns, nt_type_cols)
    score_col = resolve_column(columns, nt_score_cols)
    table = pd.read_csv(path, usecols=[root_col, type_col, score_col], dtype={root_col: 'string'}, low_memory=False)
    table = table.rename(columns={root_col: 'root_id', type_col: 'nt_type', score_col: 'nt_type_score'})
    table['root_id'] = clean_root_id(table['root_id'])
    table['nt_type'] = clean_text(table['nt_type'])
    table['nt_type_score'] = pd.to_numeric(table['nt_type_score'], errors='coerce')
    table = table.dropna(subset=['root_id']).copy()
    table = table.sort_values(['root_id', 'nt_type_score'], ascending=[True, False], na_position='last').drop_duplicates('root_id', keep='first').reset_index(drop=True)
    return table

def header_prefix_from_column(column_name):
    match = re.search('(720575940)$', str(column_name))
    if match:
        return match.group(1)
    match = re.search('root_id_(\\d{9})', str(column_name))
    if match:
        return match.group(1)
    return None

def transform_synapse_root_ids(series, mode, prefix):
    cleaned = clean_root_id(series)
    if mode == 'raw':
        return cleaned
    if mode == 'header_prefix':
        if not prefix:
            raise ValueError('header_prefix mode requires a prefix.')
        full_length = 18
        suffix_length = full_length - len(prefix)
        result = cleaned.copy()
        short_mask = result.notna() & result.str.fullmatch(f'\\d{{1,{suffix_length}}}')
        result.loc[short_mask] = prefix + result.loc[short_mask].str.zfill(suffix_length)
        return result
    raise ValueError(mode)

def select_root_id_encoding(synapse_path, pre_root_col, nt_root_ids):
    probe = pd.read_csv(synapse_path, usecols=[pre_root_col], dtype={pre_root_col: 'string'}, nrows=20000)[pre_root_col]
    candidates = [('raw', None)]
    prefix = header_prefix_from_column(pre_root_col)
    if prefix:
        candidates.append(('header_prefix', prefix))
    diagnostics = []
    for mode, prefix in candidates:
        transformed = transform_synapse_root_ids(probe, mode, prefix)
        unique = set(transformed.dropna().astype(str))
        matched = unique & nt_root_ids
        diagnostics.append({'mode': mode, 'prefix': prefix, 'unique_match_fraction': len(matched) / len(unique) if unique else 0.0, 'n_unique_probe_roots': len(unique), 'n_unique_matches': len(matched)})
    best = max(diagnostics, key=lambda row: row['unique_match_fraction'])
    return (best, diagnostics)

def apply_coordinate_transform(coordinates, mode):
    coordinates = np.asarray(coordinates, dtype=np.float64)
    if mode == 'nm':
        return coordinates
    if mode == 'voxel_4_4_40':
        return coordinates * np.array([4.0, 4.0, 40.0], dtype=np.float64)[None, :]
    raise ValueError(mode)

def select_coordinate_mode(synapse_path, coordinate_columns, tlc_points_nm, tlc_tree):
    probe = pd.read_csv(synapse_path, usecols=coordinate_columns, nrows=20000, low_memory=False)
    probe_xyz = probe[coordinate_columns].apply(pd.to_numeric, errors='coerce').dropna().to_numpy(dtype=np.float64)
    tlc_min = np.asarray(tlc_points_nm, dtype=np.float64).min(axis=0)
    tlc_max = np.asarray(tlc_points_nm, dtype=np.float64).max(axis=0)
    padding_nm = 50000.0
    diagnostics = []
    for mode in ['nm', 'voxel_4_4_40']:
        transformed = apply_coordinate_transform(probe_xyz, mode)
        inside = np.all(transformed >= (tlc_min - padding_nm)[None, :], axis=1) & np.all(transformed <= (tlc_max + padding_nm)[None, :], axis=1)
        distances_nm, _ = tlc_tree.query(transformed.astype(np.float32), k=1, workers=workers)
        diagnostics.append({'mode': mode, 'fraction_inside_trachea_bbox_plus_50um': float(inside.mean()), 'median_nearest_TLC_um': float(np.median(distances_nm) / 1000.0)})
    selected = min(diagnostics, key=lambda row: (-row['fraction_inside_trachea_bbox_plus_50um'], row['median_nearest_TLC_um']))
    return (selected['mode'], diagnostics)

def empty_hist():
    return np.zeros(n_hist_bins, dtype=np.int64)

def add_hist(hist, values):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values):
        hist += np.histogram(values, bins=hist_edges)[0]

def hist_quantile(hist, q):
    hist = np.asarray(hist, dtype=np.int64)
    total = int(hist.sum())
    if total == 0:
        return np.nan
    target = q * total
    cumulative = np.cumsum(hist)
    idx = int(np.searchsorted(cumulative, target, side='left'))
    if idx >= len(fine_edges) - 1:
        return np.nan
    previous = int(cumulative[idx - 1]) if idx > 0 else 0
    count = int(hist[idx])
    if count <= 0:
        return float(fine_edges[idx])
    fraction = (target - previous) / count
    return float(fine_edges[idx] + fraction * (fine_edges[idx + 1] - fine_edges[idx]))

def ecdf_from_hist(hist):
    hist = np.asarray(hist, dtype=np.int64)
    total = hist.sum()
    if total == 0:
        return (np.array([]), np.array([]))
    x = fine_edges[1:]
    y = np.cumsum(hist[:-1]) / total
    return (x, y)

def nt_display_name(value):
    raw = str(value).strip()
    key = re.sub('[^a-z0-9]+', '', raw.lower())
    mapping = {'ach': 'ACh', 'acetylcholine': 'ACh', 'da': 'DA', 'dopamine': 'DA', 'gaba': 'GABA', 'glut': 'GLUT', 'glutamate': 'GLUT', 'oct': 'OCT', 'octopamine': 'OCT', 'ser': 'SER', 'serotonin': 'SER', '5ht': 'SER'}
    return mapping.get(key, raw)

def tracheole_geometry(nodes):
    """Sample all CATMAID edges at 100 nm and identify distal terminal ends."""
    samples, terminal_rows = [], []
    for sid, group in nodes.groupby("skeleton_id", sort=False):
        ids = group.node_id.to_numpy(np.int64)
        coords = group[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float64)
        lookup = {int(nid): coords[i] for i, nid in enumerate(ids)}
        parents = set(group.parent_id.dropna().astype(np.int64).tolist())
        roots = group[group.parent_id.isna()]
        ends = group[~group.node_id.isin(parents) & group.parent_id.notna()]
        sections = []
        if len(roots):
            sections.append(roots[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float64))
        for row in group.itertuples(index=False):
            if pd.isna(row.parent_id):
                continue
            parent = lookup.get(int(row.parent_id))
            if parent is None:
                continue
            child = np.array([row.x_nm, row.y_nm, row.z_nm], dtype=np.float64)
            vector = child - parent
            length = float(np.linalg.norm(vector))
            if not np.isfinite(length) or length < 0:
                continue
            if length == 0:
                sections.append(child.reshape(1, 3))
                continue
            n = max(int(math.ceil(length / spacing_nm)), 1)
            fractions = np.arange(1, n + 1, dtype=np.float64) / n
            sections.append(parent[None, :] + fractions[:, None] * vector[None, :])
        if not sections:
            raise ValueError(f"Tracheole {sid} has no sampled points")
        samples.append(np.vstack(sections))
        for row in ends.itertuples(index=False):
            terminal_rows.append((int(sid), int(row.node_id), float(row.x_nm),
                                  float(row.y_nm), float(row.z_nm)))

    tracheole_points = np.vstack(samples).astype(np.float32, copy=False)
    ends = pd.DataFrame(terminal_rows, columns=["skeleton_id", "node_id", "x_nm", "y_nm", "z_nm"])
    return tracheole_points, ends


def contact_histograms(synapse_path, nt_lookup, trachea_tree, end_tree):
    """Stream presynaptic contacts and histogram their nearest terminal-end distances."""
    columns = list(pd.read_csv(synapse_path, nrows=0).columns)
    root_col = resolve_column(columns, syn_root_cols)
    xyz_cols = [resolve_column(columns, candidates) for candidates in (syn_x_cols, syn_y_cols, syn_z_cols)]
    root_ids = set(nt_lookup.root_id.dropna().astype(str))
    root_encoding, _ = select_root_id_encoding(synapse_path, root_col, root_ids)
    coordinate_mode, _ = select_coordinate_mode(synapse_path, xyz_cols, trachea_tree.data, trachea_tree)
    print(f"Coordinates: {coordinate_mode}; root IDs: {root_encoding['mode']}")

    nt_indexed = nt_lookup.set_index("root_id")[["nt_type", "nt_type_score"]]
    all_hist = empty_hist()
    nt_hist = {}
    n_contacts = n_nt = 0
    reader = pd.read_csv(synapse_path, usecols=[root_col, *xyz_cols], dtype={root_col: "string"},
                         chunksize=chunk_size, low_memory=False)
    for chunk in reader:
        raw_xyz = chunk[xyz_cols].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)
        valid = np.all(np.isfinite(raw_xyz), axis=1)
        if not valid.any():
            continue
        xyz_nm = apply_coordinate_transform(raw_xyz[valid], coordinate_mode).astype(np.float32)
        end_nm, _ = end_tree.query(xyz_nm, k=1, workers=workers)
        end_um = end_nm / 1000.0
        add_hist(all_hist, end_um)
        n_contacts += len(end_um)

        roots = transform_synapse_root_ids(chunk.loc[valid, root_col], root_encoding["mode"],
                                           root_encoding["prefix"]).reset_index(drop=True)
        annotated = pd.DataFrame({"root_id": roots}).join(nt_indexed, on="root_id")
        annotated["nt_type"] = clean_text(annotated["nt_type"])
        annotated["nt_type_score"] = pd.to_numeric(annotated["nt_type_score"], errors="coerce")
        high_conf = (annotated.nt_type.notna() & annotated.nt_type_score.ge(nt_min_score)).fillna(False).to_numpy(bool)
        if high_conf.any():
            n_nt += int(high_conf.sum())
            nt_values = annotated.loc[high_conf, "nt_type"].astype(str).to_numpy()
            end_high = end_um[high_conf]
            for nt in np.unique(nt_values):
                if str(nt) not in nt_hist:
                    nt_hist[str(nt)] = empty_hist()
                add_hist(nt_hist[str(nt)], end_high[nt_values == nt])
    print(f"Contacts: {n_contacts:,}; high-confidence NT contacts: {n_nt:,}")
    return all_hist, nt_hist


def rebin_for_test(hist):
    """Rebin 0.05-µm counts into 1-µm bins and a >=50-µm tail bin."""
    finite_counts = np.asarray(hist, np.int64)[:-1]
    left = fine_edges[:-1]
    out = np.zeros(51, dtype=np.int64)
    index = np.floor(left / 1.0).astype(int)
    regular = (index >= 0) & (index < 50)
    np.add.at(out, index[regular], finite_counts[regular])
    out[-1] += int(finite_counts[~regular].sum()) + int(hist[-1])
    return out


# Compute distances and figures
nodes = load_trachea_nodes(node_csv)
tracheole_points, ends = tracheole_geometry(nodes)
trachea_tree = cKDTree(tracheole_points)
end_tree = cKDTree(ends[["x_nm", "y_nm", "z_nm"]].to_numpy(np.float32))
all_hist, nt_hist = contact_histograms(syn_csv, load_nt_lookup(nt_csv), trachea_tree, end_tree)

nt_colors = {"ACh": "#2020df", "GABA": "#df2020", "GLUT": "#df9c20",
             "DA": "#963939", "OCT": "#107010", "SER": "#701070"}
nt_order = ["ACh", "GABA", "GLUT", "DA", "OCT", "SER"]
nt_records = sorted(((nt_display_name(key), hist) for key, hist in nt_hist.items()),
                    key=lambda row: nt_order.index(row[0]) if row[0] in nt_order else 999)
nt_records = [(name, hist) for name, hist in nt_records if name in nt_colors]

plt.rcParams.update({
    "font.family": "Arial", "font.size": 12, "axes.labelsize": 13,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5,
    "axes.linewidth": 1.0, "xtick.major.width": 1.0, "ytick.major.width": 1.0,
    "xtick.major.size": 5, "ytick.major.size": 5, "xtick.direction": "out",
    "ytick.direction": "out", "svg.fonttype": "none", "pdf.fonttype": 42, "ps.fonttype": 42,
})


def style_axes(ax):
    ax.set(xlim=(0, 45), ylim=(0, 1.01),
           xlabel="Synapse-to-nearest-terminal-end distance (µm)", ylabel="Cumulative fraction")
    ax.xaxis.labelpad = ax.yaxis.labelpad = 7
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", direction="out")
    ax.set_xticks([0, 15, 30, 45])
    ax.xaxis.set_minor_locator(MultipleLocator(5))
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0", "", "", "", "1"])
    ax.grid(False)
    fig = ax.figure
    fig.subplots_adjust(left=0.17, right=0.975, bottom=0.17, top=0.965)


# Figure 6E: all contacts
median_all = hist_quantile(all_hist, 0.5)
x, y = ecdf_from_hist(all_hist)
fig, ax = plt.subplots(figsize=(5.7, 4.8), facecolor="white")
ax.plot(x, y, color="#303030", linewidth=2, solid_capstyle="round", zorder=3)
ax.axvline(median_all, color="#777777", linestyle=(0, (3, 2.5)), linewidth=1.25, zorder=2)
ax.text(median_all, 0.955, f"{median_all:.2f} µm", ha="center", va="bottom", fontsize=12, color="#4D4D4D")
style_axes(ax)
fig.savefig(out_dir / "fig6e.svg", bbox_inches="tight", facecolor="white")
plt.show()

# Figure 6F: contacts by neurotransmitter
fig, ax = plt.subplots(figsize=(5.7, 4.8), facecolor="white")
for name, hist in nt_records:
    x, y = ecdf_from_hist(hist)
    ax.plot(x, y, color=nt_colors[name], linewidth=1.45, solid_capstyle="round", label=name)
style_axes(ax)
ax.legend(frameon=False, loc="upper left", fontsize=8.5, handlelength=1.8,
          labelspacing=0.25, borderaxespad=0.5)
fig.savefig(out_dir / "fig6f.svg", bbox_inches="tight", facecolor="white")
plt.show()

# Figure 6F: chi-square test and Cramer's V
contingency = np.vstack([rebin_for_test(hist) for _, hist in nt_records])
contingency = contingency[:, contingency.sum(axis=0) > 0]
chi2, p, df, _ = chi2_contingency(contingency, correction=False)
n_total = float(contingency.sum())
r, k = contingency.shape
cramers_v = float(np.sqrt(chi2 / (n_total * min(r - 1, k - 1))))
print(f"Figure 6E: median = {median_all:.2f} µm (n = {all_hist.sum():,})")
print(f"Figure 6F: χ²({df}) = {chi2:,.0f}, P = {p:.3g}, Cramer's V = {cramers_v:.3f}")
pd.DataFrame([{"n_contacts": int(all_hist.sum()), "median_terminal_end_um": median_all,
               "n_nt_contacts": int(n_total), "chi_square": chi2, "df": int(df),
               "p_value": p, "cramers_v": cramers_v}]).to_csv(out_dir / "fig6_stats.csv", index=False)
